# 법률 문서 검토 보조 - Colab 실행 데모

이 노트북은 `privacy-consent-review-ai` 프로젝트의 분석 기능을 Google Colab에서 재현하기 위한 제출용 파일입니다.

현재 프로젝트는 학습된 신경망 모델이나 모델 가중치를 사용하는 방식이 아니라, 공식 법령에 연결된 **결정론적 규칙 기반 AI 분석기**입니다. 따라서 별도의 `.pt`, `.pkl`, `.joblib` 모델 파일은 없습니다.

분석 결과는 법 위반이나 계약 효력을 판정하지 않으며, 누락 가능성과 추가 검토 지점을 제시합니다.

## 1. 저장소와 실행 환경 준비

Colab에서는 GitHub 저장소를 복제하고 필요한 Python 패키지를 설치합니다. 로컬 저장소에서 실행할 때는 현재 폴더를 그대로 사용합니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/sam3319/privacy-consent-review-ai.git"
PROJECT_DIR = Path("/content/privacy-consent-review-ai")

if Path("src").exists() and Path("data").exists():
    PROJECT_DIR = Path.cwd()
elif not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)],
        check=True,
    )

os.chdir(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

project_path = str(PROJECT_DIR.resolve())
if project_path in sys.path:
    sys.path.remove(project_path)
sys.path.insert(0, project_path)

print(f"프로젝트 경로: {PROJECT_DIR.resolve()}")
print("환경 준비 완료")

## 2. 분석 함수 불러오기

In [ ]:
from pprint import pprint

from src.analyzer import analyze_document
from src.contract_analyzer import analyze_contract
from src.document_classifier import detect_document_type
from src.special_contract_analyzer import (
    analyze_employment_contract,
    analyze_housing_contract,
)

def summarize_result(result):
    print(f"문서 유형: {result['document_type']}")
    print(f"핵심정보 탐지율: {result['completeness']}%")
    print("\n[구조화 추출]")
    for field in result.get("extracted_fields", []):
        value = field["value"] or "찾지 못함"
        print(f"- {field['label']}: {value} ({field['status']})")
    print("\n[검토 신호]")
    if not result["findings"]:
        print("- 현재 규칙으로 탐지된 검토 신호가 없습니다.")
    for finding in result["findings"]:
        print(
            f"- [{finding['severity']}] {finding['title']} | "
            f"{finding['article']} | {finding['status']}"
        )

print("분석 함수 로드 완료")

## 3. 문서 유형 자동 분류 확인

In [ ]:
sample_files = [
    "samples/complete_collection_consent.txt",
    "samples/risky_standard_terms_contract.txt",
    "samples/risky_housing_lease.txt",
    "samples/risky_employment_contract.txt",
]

for filename in sample_files:
    text = Path(filename).read_text(encoding="utf-8")
    detection = detect_document_type(text)
    print(
        f"{Path(filename).name}: {detection['label']} "
        f"(신뢰도 {detection['confidence']}%)"
    )

## 4. 개인정보 수집·이용 동의서 분석

In [ ]:
privacy_text = Path("samples/complete_collection_consent.txt").read_text(
    encoding="utf-8"
)
privacy_result = analyze_document(privacy_text, "collection")
summarize_result(privacy_result)

## 5. 일반 약관형 계약서 분석

약관법 관련 검토 신호와 계약 핵심정보, 조항 구조, 당사자 관점 요약을 확인합니다.

In [ ]:
contract_text = Path("samples/risky_standard_terms_contract.txt").read_text(
    encoding="utf-8"
)
contract_result = analyze_contract(contract_text, perspective="을")
summarize_result(contract_result)

print("\n[을 관점 요약]")
pprint(contract_result["perspective_summary"])

## 6. 주택 임대차·전세·월세 계약서 분석

주택임대차보호법에 연결된 계약기간, 보증금 반환, 갱신요구권, 차임 증액 등의 검토 신호를 확인합니다.

In [ ]:
housing_text = Path("samples/risky_housing_lease.txt").read_text(
    encoding="utf-8"
)
housing_result = analyze_housing_contract(housing_text, perspective="임차인")
summarize_result(housing_result)

## 7. 근로계약서 분석

근로조건 서면 명시, 근로시간, 위약금, 임금 지급 방식 및 2026년 적용 최저임금 시간급 10,320원 미달 가능성을 확인합니다.

In [ ]:
employment_text = Path("samples/risky_employment_contract.txt").read_text(
    encoding="utf-8"
)
employment_result = analyze_employment_contract(
    employment_text,
    perspective="근로자",
)
summarize_result(employment_result)

## 8. 사용자 문서 업로드 및 자동 분석

Colab에서 TXT 파일을 업로드하면 문서 유형을 자동 감지하여 적절한 분석기를 실행합니다. PDF, DOCX 및 이미지 OCR은 Streamlit 웹 앱에서 확인하는 것을 권장합니다.

In [ ]:
def analyze_text_automatically(text):
    detection = detect_document_type(text)
    document_type = detection["document_type"]

    if document_type == "standard_terms_contract":
        result = analyze_contract(text, perspective="을")
    elif document_type == "housing_lease":
        result = analyze_housing_contract(text, perspective="임차인")
    elif document_type == "employment_contract":
        result = analyze_employment_contract(text, perspective="근로자")
    else:
        result = analyze_document(text, document_type)

    print(
        f"자동 감지: {detection['label']} "
        f"(신뢰도 {detection['confidence']}%)"
    )
    summarize_result(result)
    return result

try:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        uploaded_name, uploaded_data = next(iter(uploaded.items()))
        uploaded_text = uploaded_data.decode("utf-8")
        uploaded_result = analyze_text_automatically(uploaded_text)
except ImportError:
    print("파일 업로드 셀은 Google Colab에서 사용할 수 있습니다.")

## 9. 전체 자동 테스트

저장소에 포함된 테스트를 실행하여 분석기와 Streamlit 입력 흐름을 검증합니다.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
test_result.check_returncode()

## 제출 구성

- 웹 서비스: Streamlit 배포 URL
- 실행 노트북: `privacy_consent_review_demo.ipynb`
- 전체 소스 및 법률 규칙: GitHub 저장소
- 분석 방식: 공식 법령 기반 규칙 엔진, 정규표현식, 구조화 필드 추출 및 문서 유형 점수화

본 결과는 법률 자문이나 위법 여부 판정이 아닙니다.